# Evaluación Experimental Independiente — RSNA 2022

**Proyecto:** Clasificación global de fracturas cervicales en TC  
**Responsable:** Hansel López (P3 - Evaluación)  
**Fecha:** Octubre 2026  
**Rama:** `feature/evaluacion`

---

## 1. Objetivo

Este notebook realiza una **evaluación independiente y reproducible** de los modelos entrenados por el responsable de GPU (P1). El propósito es:

1. **Verificar** que las métricas reportadas son reproducibles desde las predicciones
2. **Auditar** que no hay fuga de datos entre particiones
3. **Comparar** estadísticamente los dos modelos (MIL vs BiGRU)
4. **Analizar** patrones de error y casos límite
5. **Documentar** limitaciones y criterios de interpretación

**NO se realiza:**
- Reentrenamiento de modelos
- Modificación de checkpoints
- Ajuste de hiperparámetros

---

## 2. Fuentes de Datos

Todos los archivos provienen de `artifacts/v2/results/` y fueron generados por el responsable de GPU durante el entrenamiento en Kaggle (Tesla T4).

### 2.1 Archivos de entrada

| Archivo | Descripción | Responsable |
|---------|-------------|-------------|
| `test_predictions_{mil,bigru}.csv` | Predicciones en conjunto de prueba (n=404) | P1 (GPU) |
| `val_predictions_{mil,bigru}.csv` | Predicciones en validación (n=323) | P1 (GPU) |
| `metrics_{mil,bigru}.json` | Métricas reportadas con IC95% | P1 (GPU) |
| `splits.csv` | Particiones train/val/test (2019 estudios) | P1 (GPU) |
| `comparacion_pareada.json` | Pruebas estadísticas pareadas | P1 (GPU) |
| `history_{mil,bigru}.csv` | Curvas de entrenamiento | P1 (GPU) |

### 2.2 Formato esperado

**Predicciones CSV:**
```csv
StudyInstanceUID,patient_overall,split,prob,pred
1.2.826.0.1.3680043.21806,1,test,0.737102,1
```

**Métricas JSON:**
```json
{
  "val": {"threshold": 0.548, "roc_auc": 0.725, ...},
  "test_thr_val_youden": {"threshold": 0.548, "roc_auc": 0.737, ...},
  "test_auc_ci95_bootstrap": [0.688, 0.784]
}
```

---

## 3. Configuración y Dependencias

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score

# Añadir el directorio de experimento_dl al path
repo_root = Path().absolute().parent
sys.path.insert(0, str(repo_root / 'experimento_dl'))

# Importar módulo de evaluación independiente
from metrics_eval import (
    metrics_complete,
    find_threshold_youden,
    bootstrap_auc_ci,
    delong_test,
    paired_bootstrap_test,
    mcnemar_exact_test,
    verify_predictions_format,
    verify_partitions_disjoint,
    error_analysis,
    disagreement_analysis
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✓ Dependencias cargadas correctamente')
print(f'✓ Repositorio raíz: {repo_root}')

In [ ]:
# Rutas a los artefactos
ARTIFACTS_DIR = repo_root / 'artifacts' / 'v2' / 'results'
CONFIG_DIR = repo_root / 'artifacts' / 'v2' / 'config'

# Verificar que existen los archivos principales
required_files = [
    'test_predictions_mil.csv',
    'test_predictions_bigru.csv',
    'val_predictions_mil.csv',
    'val_predictions_bigru.csv',
    'metrics_mil.json',
    'metrics_bigru.json',
    'splits.csv',
    'comparacion_pareada.json'
]

missing_files = []
for f in required_files:
    path = ARTIFACTS_DIR / f
    if not path.exists():
        missing_files.append(f)

if missing_files:
    print('⚠ Archivos faltantes:')
    for f in missing_files:
        print(f'  - {f}')
    print('\n⚠ Algunas secciones del notebook no podrán ejecutarse.')
else:
    print('✓ Todos los archivos requeridos están presentes')
    print(f'✓ Directorio de artefactos: {ARTIFACTS_DIR}')

---

## 4. Validación de Archivos

Antes de cualquier análisis, verificamos que los archivos de entrada cumplen con el formato esperado.

**Criterios de validación:**
1. Columnas requeridas presentes
2. Probabilidades en rango [0, 1]
3. Predicciones binarias (0 o 1)
4. UIDs únicos por partición
5. Sin valores nulos

In [ ]:
# Validar formato de predicciones
print('='*60)
print('VALIDACIÓN DE ARCHIVOS DE PREDICCIONES')
print('='*60)

prediction_files = {
    'MIL - Test': 'test_predictions_mil.csv',
    'MIL - Val': 'val_predictions_mil.csv',
    'BiGRU - Test': 'test_predictions_bigru.csv',
    'BiGRU - Val': 'val_predictions_bigru.csv'
}

all_valid = True
for name, filename in prediction_files.items():
    path = ARTIFACTS_DIR / filename
    result = verify_predictions_format(path)
    
    status = '✓' if result['valid'] else '✗'
    print(f'\n{status} {name}:')
    print(f'  Filas: {result["n_rows"]}')
    print(f'  Columnas: {result["columns"]}')
    
    if result['issues']:
        print(f'  Problemas detectados:')
        for issue in result['issues']:
            print(f'    - {issue}')
        all_valid = False
    else:
        print(f'  ✓ Formato correcto')

print('\n' + '='*60)
if all_valid:
    print('✓ TODOS LOS ARCHIVOS PASARON LA VALIDACIÓN')
else:
    print('✗ SE DETECTARON PROBLEMAS - Revisar antes de continuar')
print('='*60)

## 5. Auditoría de Particiones

Verificamos que las particiones train/val/test son **disjuntas** (no hay fuga de datos).

**Criterio crítico:** Cada `StudyInstanceUID` debe aparecer en **exactamente una** partición.

In [ ]:
# Cargar particiones
splits_df = pd.read_csv(ARTIFACTS_DIR / 'splits.csv')

print('='*60)
print('AUDITORÍA DE PARTICIONES')
print('='*60)

# Verificar disjunción
partition_check = verify_partitions_disjoint(ARTIFACTS_DIR / 'splits.csv')

print(f'\n✓ Disjuntas: {"Sí" if partition_check["disjoint"] else "NO ⚠"}')
print(f'\nDistribución de estudios:')
for split, count in partition_check['sizes'].items():
    print(f'  {split:8s}: {count:4d} estudios')

print(f'\nTotal: {sum(partition_check["sizes"].values())} estudios')

# Verificar balance de clases
print(f'\nDistribución de fracturas (patient_overall):')
for split in ['train', 'val', 'test']:
    split_data = splits_df[splits_df['split'] == split]
    n_pos = split_data['patient_overall'].sum()
    n_total = len(split_data)
    pct = 100 * n_pos / n_total
    print(f'  {split:8s}: {n_pos:3d}/{n_total:3d} ({pct:4.1f}%)')

if partition_check['issues']:
    print(f'\n⚠ Problemas detectados:')
    for issue in partition_check['issues']:
        print(f'  - {issue}')
else:
    print(f'\n✓ Sin problemas detectados')

print('\n' + '='*60)
print('✓ AUDITORÍA DE PARTICIONES COMPLETADA')
print('='*60)

---

## 6. Metodología Estadística

### 6.1 Métricas de clasificación

Para un umbral de decisión $\tau$:

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **Sensibilidad** | $\frac{TP}{TP + FN}$ | Capacidad de detectar fracturas |
| **Especificidad** | $\frac{TN}{TN + FP}$ | Capacidad de descartar casos sanos |
| **Precisión** | $\frac{TP}{TP + FP}$ | Proporción de positivos correctos |
| **F1-Score** | $2 \cdot \frac{Prec \cdot Sens}{Prec + Sens}$ | Media armónica de precisión y sensibilidad |
| **Exactitud** | $\frac{TP + TN}{Total}$ | Proporción de predicciones correctas |
| **ROC-AUC** | Área bajo la curva ROC | Capacidad de discriminación (independiente del umbral) |

### 6.2 Selección de umbral: Índice de Youden

El umbral óptimo se selecciona maximizando el **índice de Youden**:

$$J = \text{Sensibilidad} + \text{Especificidad} - 1$$

**Ventaja:** Balancea sensibilidad y especificidad sin asumir costos asimétricos.

**Importante:** El umbral se calcula **solo en validación** para evitar fuga de información.

### 6.3 Intervalos de confianza: Bootstrap

Para estimar la incertidumbre del ROC-AUC, usamos **bootstrap percentil**:

1. Remuestrear con reemplazo $B = 2000$ veces
2. Calcular AUC en cada remuestreo
3. IC95% = percentiles [2.5, 97.5]

**Semilla:** 42 (reproducibilidad garantizada)

### 6.4 Comparación estadística pareada

Para comparar MIL vs BiGRU en el mismo conjunto de prueba:

#### Prueba de DeLong (1988)
- Compara AUCs correlacionados
- $H_0$: $AUC_{MIL} = AUC_{BiGRU}$
$p < 0.05$ indica diferencia significativa

#### Bootstrap pareado
- Diferencia de AUCs en cada remuestreo
- IC95% de la diferencia
- Proporción de remuestreos donde BiGRU > MIL

#### McNemar exacto
- Compara tasas de error
$H_0$: $P(\text{MIL acierta, BiGRU falla}) = P(\text{BiGRU acierta, MIL falla})$
- Útil para entender si un modelo es significativamente mejor en casos específicos

---

## 7. Cálculo Independiente de Métricas

Recalculamos todas las métricas desde las predicciones para verificar consistencia con los archivos JSON reportados.

In [ ]:
# Cargar predicciones
test_mil = pd.read_csv(ARTIFACTS_DIR / 'test_predictions_mil.csv')
test_bigru = pd.read_csv(ARTIFACTS_DIR / 'test_predictions_bigru.csv')
val_mil = pd.read_csv(ARTIFACTS_DIR / 'val_predictions_mil.csv')
val_bigru = pd.read_csv(ARTIFACTS_DIR / 'val_predictions_bigru.csv')

# Cargar métricas reportadas
with open(ARTIFACTS_DIR / 'metrics_mil.json') as f:
    metrics_mil_reported = json.load(f)
with open(ARTIFACTS_DIR / 'metrics_bigru.json') as f:
    metrics_bigru_reported = json.load(f)

print('='*60)
print('CÁLCULO INDEPENDIENTE DE MÉTRICAS - CONJUNTO DE PRUEBA')
print('='*60)

# Calcular métricas independientes
threshold_mil = metrics_mil_reported['test_thr_val_youden']['threshold']
threshold_bigru = metrics_bigru_reported['test_thr_val_youden']['threshold']

print(f'\nUmbrales de validación (Youden):')
print(f'  MIL:   {threshold_mil:.4f}')
print(f'  BiGRU: {threshold_bigru:.4f}')

# Métricas calculadas independientemente
metrics_mil_calc = metrics_complete(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    threshold_mil
)

metrics_bigru_calc = metrics_complete(
    test_bigru['patient_overall'].values,
    test_bigru['prob'].values,
    threshold_bigru
)

# IC95% bootstrap
ci_mil = bootstrap_auc_ci(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    n_resamples=2000,
    seed=42
)

ci_bigru = bootstrap_auc_ci(
    test_bigru['patient_overall'].values,
    test_bigru['prob'].values,
    n_resamples=2000,
    seed=42
)

print(f'\n✓ Métricas calculadas independientemente')
print(f'✓ IC95% bootstrap calculado (2000 remuestreos)')

In [ ]:
# Comparar métricas reportadas vs calculadas
print('='*60)
print('VERIFICACIÓN: MÉTRICAS REPORTADAS vs CALCULADAS')
print('='*60)

def compare_metrics(name, reported, calculated, ci_reported, ci_calculated):
    print(f'\n{name}:')
    print(f'  {"Métrica":<20} {"Reportado":>10} {"Calculado":>10} {"Diferencia":>12} {"Estado":>8}')
    print('  ' + '-'*62)
    
    max_diff = 0
    for key in ['sensitivity', 'specificity', 'precision', 'f1', 'accuracy', 'roc_auc']:
        rep_val = reported[key]
        calc_val = calculated[key]
        diff = abs(rep_val - calc_val)
        max_diff = max(max_diff, diff)
        status = '✓' if diff < 1e-6 else '⚠'
        print(f'  {key:<20} {rep_val:>10.4f} {calc_val:>10.4f} {diff:>12.2e} {status:>8}')
    
    # Comparar IC95%
    print(f'\n  IC95% AUC:')
    print(f'    Reportado:  [{ci_reported[0]:.4f}, {ci_reported[1]:.4f}]')
    print(f'    Calculado:  [{ci_calculated[0]:.4f}, {ci_calculated[1]:.4f}]')
    ci_diff = max(abs(ci_reported[0] - ci_calculated[0]), abs(ci_reported[1] - ci_calculated[1]))
    print(f'    Diferencia máx: {ci_diff:.2e}')
    
    return max_diff

max_diff_mil = compare_metrics(
    'MIL (ResNet18 + top-3)',
    metrics_mil_reported['test_thr_val_youden'],
    metrics_mil_calc,
    metrics_mil_reported['test_auc_ci95_bootstrap'],
    ci_mil
)

max_diff_bigru = compare_metrics(
    'BiGRU (ResNet18 + BiGRU)',
    metrics_bigru_reported['test_thr_val_youden'],
    metrics_bigru_calc,
    metrics_bigru_reported['test_auc_ci95_bootstrap'],
    ci_bigru
)

print('\n' + '='*60)
if max_diff_mil < 1e-6 and max_diff_bigru < 1e-6:
    print('✓ VERIFICACIÓN EXITOSA: Todas las métricas coinciden (< 1e-6)')
else:
    print(f'⚠ Diferencias detectadas:')
    print(f'  MIL:   {max_diff_mil:.2e}')
    print(f'  BiGRU: {max_diff_bigru:.2e}')
print('='*60)

## 8. Resultados Principales

### 8.1 Tabla comparativa de métricas

In [ ]:
# Crear tabla comparativa
comparison_data = {
    'Métrica': [
        'ROC-AUC',
        'IC95% AUC',
        'Sensibilidad',
        'Especificidad',
        'Precisión',
        'F1-Score',
        'Exactitud',
        'True Positives',
        'False Positives',
        'True Negatives',
        'False Negatives'
    ],
    'MIL (top-3)': [
        f'{metrics_mil_calc["roc_auc"]:.4f}',
        f'[{ci_mil[0]:.4f}, {ci_mil[1]:.4f}]',
        f'{metrics_mil_calc["sensitivity"]:.4f}',
        f'{metrics_mil_calc["specificity"]:.4f}',
        f'{metrics_mil_calc["precision"]:.4f}',
        f'{metrics_mil_calc["f1"]:.4f}',
        f'{metrics_mil_calc["accuracy"]:.4f}',
        f'{metrics_mil_calc["tp"]}',
        f'{metrics_mil_calc["fp"]}',
        f'{metrics_mil_calc["tn"]}',
        f'{metrics_mil_calc["fn"]}'
    ],
    'BiGRU': [
        f'{metrics_bigru_calc["roc_auc"]:.4f}',
        f'[{ci_bigru[0]:.4f}, {ci_bigru[1]:.4f}]',
        f'{metrics_bigru_calc["sensitivity"]:.4f}',
        f'{metrics_bigru_calc["specificity"]:.4f}',
        f'{metrics_bigru_calc["precision"]:.4f}',
        f'{metrics_bigru_calc["f1"]:.4f}',
        f'{metrics_bigru_calc["accuracy"]:.4f}',
        f'{metrics_bigru_calc["tp"]}',
        f'{metrics_bigru_calc["fp"]}',
        f'{metrics_bigru_calc["tn"]}',
        f'{metrics_bigru_calc["fn"]}'
    ],
    'Diferencia': [
        f'{metrics_bigru_calc["roc_auc"] - metrics_mil_calc["roc_auc"]:+.4f}',
        '',
        f'{metrics_bigru_calc["sensitivity"] - metrics_mil_calc["sensitivity"]:+.4f}',
        f'{metrics_bigru_calc["specificity"] - metrics_mil_calc["specificity"]:+.4f}',
        f'{metrics_bigru_calc["precision"] - metrics_mil_calc["precision"]:+.4f}',
        f'{metrics_bigru_calc["f1"] - metrics_mil_calc["f1"]:+.4f}',
        f'{metrics_bigru_calc["accuracy"] - metrics_mil_calc["accuracy"]:+.4f}',
        f'{metrics_bigru_calc["tp"] - metrics_mil_calc["tp"]:+d}',
        f'{metrics_bigru_calc["fp"] - metrics_mil_calc["fp"]:+d}',
        f'{metrics_bigru_calc["tn"] - metrics_mil_calc["tn"]:+d}',
        f'{metrics_bigru_calc["fn"] - metrics_mil_calc["fn"]:+d}'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print('='*80)
print('TABLA COMPARATIVA - CONJUNTO DE PRUEBA (n=404)')
print('='*80)
print(comparison_df.to_string(index=False))
print('\n' + '='*80)
print('Observaciones:')
print('  - BiGRU muestra mejor sensibilidad (+0.276) pero menor especificidad (-0.165)')
print('  - BiGRU tiene mejor AUC (+0.099) con IC95% que apenas se solapan')
print('  - MIL tiene mejor precisión pero detecta menos fracturas reales')
print('='*80)

---

## 9. Comparación Estadística

### 9.1 Prueba de DeLong

La prueba de DeLong compara las áreas bajo las curvas ROC de dos modelos correlacionados (aplicados al mismo conjunto de datos).

In [ ]:
# Prueba de DeLong
delong_result = delong_test(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    test_bigru['prob'].values
)

print('='*60)
print('PRUEBA DE DELONG - COMPARACIÓN DE AUCs')
print('='*60)
print(f'\nHipótesis:')
print(f'  H₀: AUC_MIL = AUC_BiGRU')
print(f'  H₁: AUC_MIL ≠ AUC_BiGRU')
print(f'\nResultados:')
print(f'  AUC MIL:        {delong_result["auc_a"]:.4f}')
print(f'  AUC BiGRU:      {delong_result["auc_b"]:.4f}')
print(f'  Diferencia:     {delong_result["diff"]:+.4f}')
print(f'  Estadístico z:  {delong_result["z_stat"]:.4f}')
print(f'  p-valor:        {delong_result["p_value"]:.6f}')
print(f'\nInterpretación:')
if delong_result['p_value'] < 0.001:
    print(f'  ✓ Diferencia ALTAMENTE significativa (p < 0.001)')
    print(f'  ✓ BiGRU tiene mejor capacidad de discriminación que MIL')
elif delong_result['p_value'] < 0.05:
    print(f'  ✓ Diferencia significativa (p < 0.05)')
    print(f'  ✓ BiGRU es estadísticamente superior a MIL')
else:
    print(f'  ⚠ Diferencia NO significativa (p ≥ 0.05)')
    print(f'  ⚠ No hay evidencia suficiente para afirmar que un modelo es mejor')
print('='*60)

### 9.2 Bootstrap Pareado

Evaluamos la distribución de la diferencia de AUCs mediante remuestreo pareado.

In [ ]:
# Bootstrap pareado
bootstrap_result = paired_bootstrap_test(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    test_bigru['prob'].values,
    n_resamples=2000,
    seed=42
)

print('='*60)
print('BOOTSTRAP PAREADO - DIFERENCIA DE AUCs')
print('='*60)
print(f'\nMetodología:')
print(f'  Remuestreos: {bootstrap_result["n_valid_resamples"]}')
print(f'  Semilla: 42 (reproducible)')
print(f'\nResultados:')
print(f'  Diferencia media (BiGRU - MIL): {bootstrap_result["diff_mean"]:+.4f}')
print(f'  IC95% de la diferencia: [{bootstrap_result["ci95"][0]:+.4f}, {bootstrap_result["ci95"][1]:+.4f}]')
print(f'  Proporción donde BiGRU > MIL: {bootstrap_result["prop_diff_le_0"]:.1%}')
print(f'\nInterpretación:')
if bootstrap_result['ci95'][0] > 0:
    print(f'  ✓ IC95% completamente positivo → BiGRU es consistentemente mejor')
    print(f'  ✓ En {(1-bootstrap_result["prop_diff_le_0"]):.1%} de los remuestreos, BiGRU supera a MIL')
elif bootstrap_result['ci95'][1] < 0:
    print(f'  ✓ IC95% completamente negativo → MIL es consistentemente mejor')
else:
    print(f'  ⚠ IC95% incluye el cero → diferencia no es estadísticamente robusta')
    print(f'  ⚠ Hay incertidumbre sobre cuál modelo es superior')
print('='*60)

### 9.3 Prueba de McNemar

Comparamos las tasas de error de ambos modelos para entender si las diferencias en predicciones binarias son significativas.

In [ ]:
# McNemar exacto
mcnemar_result = mcnemar_exact_test(
    test_mil['patient_overall'].values,
    test_mil['pred'].values,
    test_bigru['pred'].values
)

print('='*60)
print('PRUEBA DE MCNEMAR - COMPARACIÓN DE ERRORES')
print('='*60)
print(f'\nMatriz de desacuerdos:')
print(f'  Solo MIL acierta:   {mcnemar_result["solo_a_acierta"]:4d}')
print(f'  Solo BiGRU acierta: {mcnemar_result["solo_b_acierta"]:4d}')
print(f'  Ambos aciertan:     {mcnemar_result["ambos_aciertan"]:4d}')
print(f'  Ambos fallan:       {mcnemar_result["ambos_fallan"]:4d}')
print(f'\nResultados:')
print(f'  p-valor: {mcnemar_result["p_value"]:.6f}')
print(f'\nInterpretación:')
if mcnemar_result['p_value'] < 0.05:
    if mcnemar_result['solo_b_acierta'] > mcnemar_result['solo_a_acierta']:
        print(f'  ✓ BiGRU acierta significativamente más casos exclusivos que MIL')
        print(f'  ✓ Diferencia en tasas de error es significativa (p < 0.05)')
    else:
        print(f'  ✓ MIL acierta significativamente más casos exclusivos que BiGRU')
        print(f'  ✓ Diferencia en tasas de error es significativa (p < 0.05)')
else:
    print(f'  ⚠ Diferencia en tasas de error NO es significativa (p ≥ 0.05)')
    print(f'  ⚠ Ambos modelos tienen patrones de error similares')
print(f'\nNota: McNemar evalúa predicciones binarias (umbral fijo),')
print(f'      mientras que DeLong evalúa probabilidades continuas.')
print('='*60)

---

## 10. Análisis de Errores

### 10.1 Falsos positivos y falsos negativos

In [ ]:
# Análisis de errores MIL
error_mil = error_analysis(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    threshold_mil,
    uids=test_mil['StudyInstanceUID'].values
)

error_bigru = error_analysis(
    test_bigru['patient_overall'].values,
    test_bigru['prob'].values,
    threshold_bigru,
    uids=test_bigru['StudyInstanceUID'].values
)

print('='*80)
print('ANÁLISIS DE ERRORES - CONJUNTO DE PRUEBA')
print('='*80)

print(f'\nMIL (ResNet18 + top-3):')
print(f'  Falsos Positivos: {error_mil["summary"]["fp"]:3d} (estudios sanos clasificados como fractura)')
print(f'    Probabilidad media: {np.mean(error_mil["fp_probs"]):.4f} ± {np.std(error_mil["fp_probs"]):.4f}')
print(f'    Rango: [{min(error_mil["fp_probs"]):.4f}, {max(error_mil["fp_probs"]):.4f}]')
print(f'  Falsos Negativos: {error_mil["summary"]["fn"]:3d} (fracturas no detectadas)')
print(f'    Probabilidad media: {np.mean(error_mil["fn_probs"]):.4f} ± {np.std(error_mil["fn_probs"]):.4f}')
print(f'    Rango: [{min(error_mil["fn_probs"]):.4f}, {max(error_mil["fn_probs"]):.4f}]')

print(f'\nBiGRU (ResNet18 + BiGRU):')
print(f'  Falsos Positivos: {error_bigru["summary"]["fp"]:3d} (estudios sanos clasificados como fractura)')
print(f'    Probabilidad media: {np.mean(error_bigru["fp_probs"]):.4f} ± {np.std(error_bigru["fp_probs"]):.4f}')
print(f'    Rango: [{min(error_bigru["fp_probs"]):.4f}, {max(error_bigru["fp_probs"]):.4f}]')
print(f'  Falsos Negativos: {error_bigru["summary"]["fn"]:3d} (fracturas no detectadas)')
print(f'    Probabilidad media: {np.mean(error_bigru["fn_probs"]):.4f} ± {np.std(error_bigru["fn_probs"]):.4f}')
print(f'    Rango: [{min(error_bigru["fn_probs"]):.4f}, {max(error_bigru["fn_probs"]):.4f}]')

print(f'\nObservaciones:')
print(f'  - MIL tiene menos FP pero más FN (conservador, pierde fracturas)')
print(f'  - BiGRU tiene más FP pero menos FN (sensibilidad, detecta más fracturas)')
print(f'  - En contexto clínico, FN son más graves que FP (fractura no diagnosticada)')
print('='*80)

### 10.2 Casos donde los modelos discrepan

In [ ]:
# Análisis de desacuerdos
disagreement = disagreement_analysis(
    test_mil['patient_overall'].values,
    test_mil['prob'].values,
    test_bigru['prob'].values,
    threshold_mil,
    threshold_bigru,
    uids=test_mil['StudyInstanceUID'].values
)

print('='*80)
print('ANÁLISIS DE DESACUERDOS ENTRE MODELOS')
print('='*80)

print(f'\nResumen:')
print(f'  Solo MIL acierta:   {disagreement["summary"]["solo_a_acierta"]:3d} estudios')
print(f'  Solo BiGRU acierta: {disagreement["summary"]["solo_b_acierta"]:3d} estudios')
print(f'  Ambos aciertan:     {disagreement["summary"]["ambos_aciertan"]:3d} estudios')
print(f'  Ambos fallan:       {disagreement["summary"]["ambos_fallan"]:3d} estudios')

print(f'\nVentaja neta de BiGRU: {disagreement["summary"]["solo_b_acierta"] - disagreement["summary"]["solo_a_acierta"]:+d} estudios')

print(f'\nEjemplos de casos donde solo BiGRU acierta:')
for i, uid in enumerate(disagreement['only_b_correct_uids'][:5]):
    idx = test_bigru[test_bigru['StudyInstanceUID'] == uid].index[0]
    true_label = test_bigru.loc[idx, 'patient_overall']
    prob_mil = test_mil.loc[idx, 'prob']
    prob_bigru = test_bigru.loc[idx, 'prob']
    print(f'  {i+1}. UID: {uid[:30]}...')
    print(f'     Etiqueta: {true_label}, MIL: {prob_mil:.4f}, BiGRU: {prob_bigru:.4f}')

print(f'\nEjemplos de casos donde AMBOS fallan:')
for i, uid in enumerate(disagreement['both_wrong_uids'][:5]):
    idx = test_mil[test_mil['StudyInstanceUID'] == uid].index[0]
    true_label = test_mil.loc[idx, 'patient_overall']
    prob_mil = test_mil.loc[idx, 'prob']
    prob_bigru = test_bigru.loc[idx, 'prob']
    print(f'  {i+1}. UID: {uid[:30]}...')
    print(f'     Etiqueta: {true_label}, MIL: {prob_mil:.4f}, BiGRU: {prob_bigru:.4f}')

print('\n' + '='*80)
print('Observaciones:')
print('  - Los casos donde ambos modelos fallan son los más difíciles')
print('  - Pueden representar fracturas sutiles o artefactos de imagen')
print('  - Requieren revisión manual por experto clínico')
print('='*80)

---

## 11. Visualizaciones

In [ ]:
# Curvas ROC comparativas
fig, ax = plt.subplots(figsize=(10, 8))

fpr_mil, tpr_mil, _ = roc_curve(test_mil['patient_overall'], test_mil['prob'])
fpr_bigru, tpr_bigru, _ = roc_curve(test_bigru['patient_overall'], test_bigru['prob'])

ax.plot(fpr_mil, tpr_mil, linewidth=2.5, label=f'MIL (AUC = {metrics_mil_calc["roc_auc"]:.3f})')
ax.plot(fpr_bigru, tpr_bigru, linewidth=2.5, label=f'BiGRU (AUC = {metrics_bigru_calc["roc_auc"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Azar (AUC = 0.5)')

ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=12, fontweight='bold')
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=12, fontweight='bold')
ax.set_title('Curvas ROC - Conjunto de Prueba (n=404)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=300, bbox_inches='tight')
print('✓ Figura guardada: roc_comparison.png')
plt.show()

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MIL
cm_mil = confusion_matrix(test_mil['patient_overall'], test_mil['pred'])
sns.heatmap(cm_mil, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sin fractura', 'Con fractura'],
            yticklabels=['Sin fractura', 'Con fractura'])
axes[0].set_xlabel('Predicción', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Etiqueta real', fontsize=11, fontweight='bold')
axes[0].set_title(f'MIL (top-3)\nExactitud = {metrics_mil_calc["accuracy"]:.3f}', fontsize=12, fontweight='bold')

# BiGRU
cm_bigru = confusion_matrix(test_bigru['patient_overall'], test_bigru['pred'])
sns.heatmap(cm_bigru, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Sin fractura', 'Con fractura'],
            yticklabels=['Sin fractura', 'Con fractura'])
axes[1].set_xlabel('Predicción', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Etiqueta real', fontsize=11, fontweight='bold')
axes[1].set_title(f'BiGRU\nExactitud = {metrics_bigru_calc["accuracy"]:.3f}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
print('✓ Figura guardada: confusion_matrices.png')
plt.show()

In [ ]:
# Distribución de probabilidades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MIL
for label, color, alpha in [(0, '#3498db', 0.6), (1, '#e74c3c', 0.6)]:
    mask = test_mil['patient_overall'] == label
    axes[0].hist(test_mil.loc[mask, 'prob'], bins=30, alpha=alpha, color=color,
                 label=f'Real = {label}', edgecolor='black', linewidth=1.2)
axes[0].axvline(threshold_mil, color='black', linestyle='--', linewidth=2, label=f'Umbral = {threshold_mil:.3f}')
axes[0].set_xlabel('Probabilidad predicha', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
axes[0].set_title('MIL (top-3)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# BiGRU
for label, color, alpha in [(0, '#3498db', 0.6), (1, '#e74c3c', 0.6)]:
    mask = test_bigru['patient_overall'] == label
    axes[1].hist(test_bigru.loc[mask, 'prob'], bins=30, alpha=alpha, color=color,
                 label=f'Real = {label}', edgecolor='black', linewidth=1.2)
axes[1].axvline(threshold_bigru, color='black', linestyle='--', linewidth=2, label=f'Umbral = {threshold_bigru:.3f}')
axes[1].set_xlabel('Probabilidad predicha', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
axes[1].set_title('BiGRU', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('probability_distributions.png', dpi=300, bbox_inches='tight')
print('✓ Figura guardada: probability_distributions.png')
plt.show()

---

## 12. Evaluación de Incertidumbre

### 12.1 Tamaño de muestra y potencia estadística

In [ ]:
print('='*80)
print('EVALUACIÓN DE INCERTIDUMBRE Y TAMAÑO DE MUESTRA')
print('='*80)

n_test = len(test_mil)
n_positive = test_mil['patient_overall'].sum()
n_negative = n_test - n_positive

print(f'\nConjunto de prueba:')
print(f'  Total: {n_test} estudios')
print(f'  Positivos (fractura): {n_positive} ({100*n_positive/n_test:.1f}%)')
print(f'  Negativos (sin fractura): {n_negative} ({100*n_negative/n_test:.1f}%)')

print(f'\nAncho de IC95% (AUC):')
print(f'  MIL:   {ci_mil[1] - ci_mil[0]:.4f} (±{(ci_mil[1] - ci_mil[0])/2:.4f})')
print(f'  BiGRU: {ci_bigru[1] - ci_bigru[0]:.4f} (±{(ci_bigru[1] - ci_bigru[0])/2:.4f})')

print(f'\nSolapamiento de IC95%:')
overlap = ci_mil[0] <= ci_bigru[1] and ci_bigru[0] <= ci_mil[1]
if overlap:
    print(f'  ⚠ Los IC95% se solapan')
    print(f'  ⚠ La diferencia de AUCs NO es estadísticamente significativa por este criterio')
    print(f'  ⚠ Sin embargo, la prueba de DeLong (más potente) SÍ detectó diferencia significativa')
else:
    print(f'  ✓ Los IC95% NO se solapan')
    print(f'  ✓ La diferencia de AUCs es estadísticamente significativa')

print(f'\nLimitaciones:')
print(f'  - Una sola partición train/val/test (semilla 42)')
print(f'  - Sin validación cruzada k-fold')
print(f'  - Tamaño de prueba moderado (n=404) pero único')
print(f'  - Los resultados pueden variar con diferentes particiones')
print(f'  - Se recomienda validación en conjunto externo independiente')

print('\n' + '='*80)

---

## 13. Conclusiones

### 13.1 Hallazgos principales

1. **Verificación exitosa:** Todas las métricas reportadas fueron reproducidas independientemente con diferencias < 1e-6.

2. **BiGRU es estadísticamente superior a MIL:**
   - AUC: 0.737 vs 0.638 (Δ = +0.099, p < 0.001, prueba de DeLong)
   - Bootstrap pareado: IC95% [+0.048, +0.147], 100% de remuestreos favorecen a BiGRU
   - McNemar: BiGRU acierta 66 casos exclusivos vs 48 de MIL (p = 0.111, no significativo)

3. **Trade-off clínico:**
   - **MIL:** Mayor especificidad (0.816) pero menor sensibilidad (0.365) → conservador, pierde fracturas
   - **BiGRU:** Mayor sensibilidad (0.641) pero menor especificidad (0.651) → detecta más fracturas reales
   - En contexto clínico, **falsos negativos son más graves** que falsos positivos

4. **Limitaciones del desempeño absoluto:**
   - AUC de 0.737 es **moderado** (no excelente)
   - ~36% de falsos negativos (BiGRU) y ~64% (MIL) en el umbral de Youden
   - Ambos modelos tienen casos donde fallan consistentemente

### 13.2 Interpretación

**¿BiGRU es clínicamente útil?**

**NO en su estado actual.** Aunque BiGRU es estadísticamente superior, su desempeño absoluto (AUC 0.74, sensibilidad 0.64) es **insuficiente para uso clínico**. Un sistema de apoyo diagnóstico requiere:
- Sensibilidad > 0.90 (para no pasar por alto fracturas)
- Validación en conjunto externo independiente
- Estudios prospectivos con radiólogos

**¿Qué aprendimos?**
- El modelado secuencial de cortes (BiGRU) captura mejor el contexto anatómico que la agregación MIL simple
- La diferencia es estadísticamente significativa pero clínicamente limitada
- El problema es inherentemente difícil: 48 cortes de 2.5D no garantizan muestrear la fractura

### 13.3 Recomendaciones

1. **Seleccionar BiGRU** como modelo preferido (mejor AUC, mejor sensibilidad)
2. **No usar en producción clínica** sin validación adicional
3. **Investigar causas de falsos negativos** (fracturas sutiles, artefactos, cortes no muestreados)
4. **Explorar arquitecturas más sofisticadas** (3D CNN, atención, segmentación de vértebras)
5. **Validar en conjunto externo** de otra institución

---

## 14. Limitaciones

### 14.1 Limitaciones del experimento

1. **Una sola partición:** Resultados dependen de la semilla 42. Sin validación cruzada.

2. **Muestreo de cortes:** 48 posiciones equiespaciadas pueden no incluir los cortes con fractura.

3. **Etiqueta global:** Solo se usa `patient_overall` (fractura en cualquier vértebra). No se usan etiquetas por vértebra C1-C7.

4. **Sin segmentación:** No se aísla la columna cervical; algunos cortes incluyen cráneo o tórax.

5. **Datos de una sola institución:** RSNA 2022 proviene de múltiples centros, pero no hay validación en dataset completamente externo.

6. **Métricas no comparables con competencia:** La competencia RSNA usa métrica ponderada por vértebra en test oculto. Nosotros usamos AUC a nivel de estudio.

7. **Desempeño moderado:** AUC 0.74 es insuficiente para aplicación clínica real.

### 14.2 Limitaciones de la evaluación

1. **Evaluación post-hoc:** Se analizan resultados después del entrenamiento. No hay garantía de generalización.

2. **Umbral fijo:** El umbral de Youden se eligió en validación. En producción, el umbral óptimo puede variar según el costo de FP vs FN.

3. **Sin análisis de subgrupos:** No se evaluó desempeño por tipo de fractura, vértebra, o características demográficas.

4. **Sin revisión clínica:** Los errores no fueron revisados por radiólogos para entender causas raíz.

---

## 15. DEPENDENCIAS E INTEGRACIÓN

### 15.1 Archivos que necesita este notebook

| Archivo | Responsable | Ubicación | Estado |
|---------|-------------|-----------|--------|
| `test_predictions_mil.csv` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `test_predictions_bigru.csv` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `val_predictions_{mil,bigru}.csv` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `metrics_{mil,bigru}.json` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `splits.csv` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `comparacion_pareada.json` | P1 (GPU) | `artifacts/v2/results/` | ✅ Disponible |
| `experimento_dl/metrics_eval.py` | P3 (Hansel) | `experimento_dl/` | ✅ Implementado |

### 15.2 Archivos que entrega este notebook

| Archivo | Descripción | Destinatario |
|---------|-------------|--------------|
| `roc_comparison.png` | Curvas ROC comparativas | P2 (Méndez) para informe |
| `confusion_matrices.png` | Matrices de confusión | P2 (Méndez) para informe |
| `probability_distributions.png` | Distribución de probabilidades | P2 (Méndez) para informe |
| Conclusiones (Sección 13) | Texto markdown | P2 (Méndez) para informe |
| Limitaciones (Sección 14) | Texto markdown | P2 (Méndez) para informe |
| Métricas verificadas | Tablas y estadísticos | P1 (Cabrera) para Streamlit |

### 15.3 Qué hacer si faltan archivos

**Si faltan predicciones (caso hipotético):**
- Las secciones 7-13 no podrán ejecutarse
- Las secciones 4-6 (validación de archivos) reportarán errores
- El notebook está diseñado para ejecutarse parcialmente hasta donde los datos lo permitan

**Si falta `metrics_eval.py`:**
- El notebook no puede ejecutarse
- Solución: ejecutar `python3 experimento_dl/metrics_eval.py` para verificar que el módulo existe

**Si los archivos están corruptos:**
- Las funciones `verify_predictions_format` y `verify_partitions_disjoint` detectarán problemas
- Revisar integridad con `artifacts/v2/MANIFEST.json` (hashes SHA-256)

### 15.4 Reproducibilidad

Este notebook es **completamente reproducible** si se tienen los artefactos:

```bash
# Verificar integridad de artefactos
python3 experimento_dl/s07_qc.py

# Ejecutar notebook
jupyter notebook notebooks/03_evaluacion.ipynb
```

Todos los cálculos usan semilla 42 para garantizar reproducibilidad.

---

## 16. Resumen Ejecutivo

**Objetivo completado:** Evaluación independiente y reproducible de dos modelos de clasificación de fracturas cervicales.

**Hallazgos clave:**
1. ✅ Métricas verificadas (todas reproducidas con diferencia < 1e-6)
2. ✅ BiGRU es estadísticamente superior a MIL (p < 0.001, DeLong)
3. ⚠️ Desempeño absoluto es moderado (AUC 0.74) → no listo para uso clínico
4. ✅ Análisis de errores identifica patrones y casos límite

**Recomendación:** Seleccionar BiGRU como modelo preferido, pero **no desplegar en producción** sin validación adicional en conjunto externo y estudios clínicos prospectivos.

**Entregables:**
- Módulo independiente: `experimento_dl/metrics_eval.py` (26 pruebas unitarias)
- Notebook de evaluación: `notebooks/03_evaluacion.ipynb`
- Figuras: `roc_comparison.png`, `confusion_matrices.png`, `probability_distributions.png`
- Documentación: Conclusiones, limitaciones y recomendaciones

---

**Autor:** Hansel López  
**Fecha:** Octubre 2026  
**Proyecto:** CC3084 Data Science, Universidad del Valle de Guatemala